## N-HiTS Model
 One global model per season trained jointly on all 27 Brazilian states


### 1. Environment

In [1]:
%pip install optuna epiweeks -q scipy seaborn matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import neuralforecast
print(neuralforecast.__version__)  # should be 1.7+

c:\Users\betty\Downloads\NHiTS_Model\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-09 21:46:02,615	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-09 21:46:02,963	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


3.1.9


In [3]:
import logging
for _name in [
    'pytorch_lightning', 'pytorch_lightning.utilities.rank_zero',
    'pytorch_lightning.accelerators.cuda', 'pytorch_lightning.trainer',
    'lightning_fabric', 'lightning_fabric.utilities.seed',
    'lightning.pytorch', 'lightning.pytorch.utilities.rank_zero',
    'lightning.pytorch.accelerators', 'lightning.pytorch.trainer',
]:
    logging.getLogger(_name).setLevel(logging.ERROR)

import os, warnings
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import optuna
import torch

from utils    import REGIONS, state_order, TARGET, H
from train    import run_optuna, run_all_seasons
from plotting import plot_season, plot_wis_heatmap

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
pl.seed_everything(SEED, workers=True)

os.makedirs('outputs/plots',       exist_ok=True)
os.makedirs('outputs/scores',      exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

GPU available: False


## 2. Configuration

In [4]:
VAL_SEASONS     = ['2022-2023', '2023-2024', '2024-2025']
FORECAST_SEASON = '2025-2026'
ALL_SEASONS     = VAL_SEASONS + [FORECAST_SEASON]

# Climate covariates + cyclical epiweek features (sin/cos)
FEATURES = [
    'sin_ew', 'cos_ew',
    'temp_med_lag_0', 'rel_humid_med_lag_0',
    'temp_med_lag_8', 'rel_humid_med_lag_8', 'precip_med_lag_8',
    'enso', 'iod', 'pdo',
]

TEST_MODE = True
if TEST_MODE:
    ACTIVE_STATES  = ['AM'] #CE', 'RS', 'GO'
    OPTUNA_TRIALS  = 5
    print(f'TEST MODE: {ACTIVE_STATES}')
else:
    ACTIVE_STATES  = state_order
    OPTUNA_TRIALS  = 30
    print(f'FULL RUN: {len(ACTIVE_STATES)} states, {OPTUNA_TRIALS} Optuna trials')

print(f'Features ({len(FEATURES)}): {FEATURES}')

TEST MODE: ['AM']
Features (10): ['sin_ew', 'cos_ew', 'temp_med_lag_0', 'rel_humid_med_lag_0', 'temp_med_lag_8', 'rel_humid_med_lag_8', 'precip_med_lag_8', 'enso', 'iod', 'pdo']


## 3. Load Data

In [5]:
df_hist = pd.read_csv('data/Hist_dataset.csv')
df_fcst = pd.read_csv('data/forecast_dataset.csv')

df_hist = df_hist[df_hist['uf'].isin(ACTIVE_STATES)].copy()
df_fcst = df_fcst[df_fcst['uf'].isin(ACTIVE_STATES)].copy()

# Validate features are present
missing_hist = [f for f in FEATURES if f not in df_hist.columns]
missing_fcst = [f for f in FEATURES if f not in df_fcst.columns]
assert not missing_hist, f'Missing from df_hist: {missing_hist}'
assert not missing_fcst, f'Missing from df_fcst: {missing_fcst}'

print(f'Historical : {df_hist.shape[0]:,} rows')
print(f'Forecast   : {df_fcst.shape[0]:,} rows')
print(f'States     : {sorted(df_hist["uf"].unique())}')

Historical : 845 rows
Forecast   : 52 rows
States     : ['AM']


## 4. Optuna Hyperparameter Search

In [ ]:
best_params, study = run_optuna(
    df_hist         = df_hist,
    df_fcst         = df_fcst,
    state_order     = ACTIVE_STATES,
    val_seasons     = VAL_SEASONS,
    forecast_season = FORECAST_SEASON,
    features        = FEATURES,
    n_trials        = OPTUNA_TRIALS,
    seed            = SEED,
)

# Save Optuna results
study.trials_dataframe().to_csv('outputs/scores/optuna_trials.csv', index=False)
print('Saved: outputs/scores/optuna_trials.csv')

In [12]:
# Override best_params manually
best_params = {
     'input_size'   : 13,
     'n_stacks'     : 3,
     'mlp_units'    : 256,
     'learning_rate': 0.001,
     'batch_size'   : 64,
     'max_steps'    : 1000,
 }
print('Best params:', best_params)

Best params: {'input_size': 13, 'n_stacks': 3, 'mlp_units': 256, 'learning_rate': 0.001, 'batch_size': 64, 'max_steps': 1000}


## 5. Train All Seasons

In [13]:
all_preds, all_scores = run_all_seasons(
    df_hist         = df_hist,
    df_fcst         = df_fcst,
    state_order     = ACTIVE_STATES,
    all_seasons     = ALL_SEASONS,
    val_seasons     = VAL_SEASONS,
    forecast_season = FORECAST_SEASON,
    features        = FEATURES,
    best_params     = best_params,
    seed            = SEED,
)


  Season: 2022-2023 [VALIDATION]
  Train rows: 678  |  states: 1
  [AM] WIS=29.0  CRPS=38.8  LogS=0.62

  Season: 2023-2024 [VALIDATION]
  Train rows: 730  |  states: 1
  [AM] WIS=42.8  CRPS=60.9  LogS=0.62

  Season: 2024-2025 [VALIDATION]
  Train rows: 782  |  states: 1
  [AM] WIS=152.6  CRPS=171.5  LogS=15.11

  Season: 2025-2026 [FORECAST]
  Train rows: 835  |  states: 1
  [AM] peak median: 1390

Done. 4 predictions stored.


## 6. Scores Summary

In [14]:
df_scores = pd.DataFrame(all_scores)

print('=== Mean Scores by Season ===')
print(df_scores.groupby('season')[['WIS','CRPS','LogS','MAE','RMSE']]
      .mean().round(2).to_string())

print('\n=== Mean Scores by State ===')
print(df_scores.groupby('state')[['WIS','CRPS','LogS','MAE','RMSE']]
      .mean().round(2).to_string())

=== Mean Scores by Season ===
              WIS    CRPS   LogS     MAE    RMSE
season                                          
2022-2023   28.98   38.79   0.62   50.91   66.16
2023-2024   42.77   60.88   0.62   83.64  101.66
2024-2025  152.64  171.51  15.11  194.66  292.38

=== Mean Scores by State ===
        WIS  CRPS  LogS     MAE   RMSE
state                                 
AM     74.8  90.4  5.45  109.74  153.4


## 7. Plots

In [ ]:
for season in VAL_SEASONS:
    plot_season(
        season, all_preds, df_hist, df_scores, ACTIVE_STATES,
        is_forecast=False,
        save_path=f'outputs/plots/NHITs_Validation_{season}.png',
    )

plot_season(
    FORECAST_SEASON, all_preds, df_hist, df_scores, ACTIVE_STATES,
    is_forecast=True,
    save_path=f'outputs/plots/NHITs_Forecast_{FORECAST_SEASON}.png',
)

plot_wis_heatmap(
    df_scores, ACTIVE_STATES,
    save_path='outputs/plots/NHITs_WIS_Heatmap.png',
)

## 8. Save Outputs

In [ ]:
PRED_COLS = ['uf', 'season', 'pred',
             'lower_50', 'upper_50', 'lower_80', 'upper_80',
             'lower_90', 'upper_90', 'lower_95', 'upper_95', 'date']

# Validation scores
df_scores.to_csv('outputs/scores/NHITs_validation_scores.csv', index=False)
print('Saved: outputs/scores/NHITs_validation_scores.csv')

# Validation predictions
for season in VAL_SEASONS:
    rows = [all_preds[(s, season)] for s in ACTIVE_STATES
            if (s, season) in all_preds]
    if rows:
        out = (pd.concat(rows)
               .sort_values(['uf', 'date'])
               .reset_index(drop=True))
        out[[c for c in PRED_COLS if c in out.columns]].to_csv(
            f'outputs/predictions/NHITs_validation_{season}.csv', index=False
        )
        print(f'Saved: outputs/predictions/NHITs_validation_{season}.csv')

# Forecast predictions
fc_rows = [all_preds[(s, FORECAST_SEASON)] for s in ACTIVE_STATES
           if (s, FORECAST_SEASON) in all_preds]
if fc_rows:
    fc_df = (pd.concat(fc_rows)
             .sort_values(['uf', 'date'])
             .reset_index(drop=True))
    fc_df[[c for c in PRED_COLS if c in fc_df.columns]].to_csv(
        f'outputs/predictions/NHITs_forecast_{FORECAST_SEASON}.csv', index=False
    )
    print(f'Saved: outputs/predictions/NHITs_forecast_{FORECAST_SEASON}.csv')

print('Done.')